In [2]:
# For working with data
import pandas as pd
import numpy as np

# For visualization
import seaborn as sns
import matplotlib.pyplot as plt

# For machine learning modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.impute import SimpleImputer

In [3]:
filepath = "../data/raw/train.csv"
raw = pd.read_csv(filepath)

raw.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Customer Status,Last Due Date,Last Payment Date,Churn
0,160831,34,Male,NaN,26,none,17.0,Basic,Quarterly,890.21,30.0,active,06-21,07-08,0
1,354435,30,Male,36.0,22,NaN,15.0,Basic,Annual,954.37,30.0,active,06-10,06-25,0
2,444576,23,Male,44.0,14,none,NaN,Premium,Annual,674.00,21.0,inactive,06-27,07-21,1
3,130847,24,Male,39.0,21,2,14.0,Standard,Quarterly,775.00,18.0,active,06-30,07-14,0
4,108121,26,Male,NaN,26,1,5.0,Basic,Quarterly,679.45,NaN,active,06-29,07-04,0


In [4]:
raw.describe()

,CustomerID,Age,Tenure,Usage Frequency,Payment Delay,Total Spend,Last Interaction,Churn
count,303124.000000,303124.000000,241103.000000,303124.000000,214800.000000,303124.000000,240574.000000,303124.000000
mean,252365.776715,38.444178,31.530657,15.954088,10.258073,667.087312,14.051415,0.299999
std,145860.674838,11.669331,17.206895,8.536377,6.564950,223.225242,8.526133,0.458258
min,1.000000,18.000000,1.000000,1.000000,0.000000,100.000000,1.000000,0.000000
25%,125964.750000,29.000000,17.000000,9.000000,5.000000,540.190000,7.000000,0.000000
50%,252101.500000,39.000000,32.000000,16.000000,10.000000,693.835000,13.000000,0.000000
75%,378554.250000,47.000000,46.000000,23.000000,15.000000,846.532500,21.000000,1.000000
max,505205.000000,65.000000,60.000000,30.000000,30.000000,1000.000000,30.000000,1.000000


## Deal with missing Values

In [5]:
raw.isna().sum()

CustomerID               0
Age                      0
Gender                   0
Tenure               62021
Usage Frequency          0
Support Calls        62420
Payment Delay        88324
Subscription Type        0
Contract Length          0
Total Spend              0
Last Interaction     62550
Customer Status          0
Last Due Date            0
Last Payment Date        0
Churn                    0
dtype: int64

In [6]:
# One hot encoding Categorical values

raw['Subscription Type'] = raw['Subscription Type'].replace({
    'Basic': 0,
    'Standard': 1,
    'Premium': 2
    })

raw['Contract Length'] = raw['Contract Length'].replace({
    'Monthly': 0,
    'Quarterly': 1,
    'Annual': 2
    })

raw['Customer Status'] = raw['Customer Status'].replace({
    'active': 1,
    'inactive': 0
    })

raw['Gender'] = raw['Gender'].replace({
    'Male': 1,
    'Female': 0
    })

raw['Support Calls'] = pd.to_numeric(raw['Support Calls'].replace('none', 0))



In [7]:
# Convert date columns to datetime format
raw['Last Due Date'] = pd.to_datetime(raw['Last Due Date'], format="%m-%d")
raw['Last Payment Date'] = pd.to_datetime(raw['Last Payment Date'], format="%m-%d")


# Fill in missing Payment Delay values by subtracting Last Payment Date with Last Due Date
raw['Payment Delay'] = (raw['Last Payment Date'] - raw['Last Due Date']).dt.days

C:\Users\stoya\AppData\Local\Temp\ipykernel_23956\1694970456.py:2: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  raw['Last Due Date'] = pd.to_datetime(raw['Last Due Date'], format="%m-%d")


### I will use a simple imputer to fill in the rest of the missing values with medians of the column

In [8]:
#Impute missing values using 3-NN
impute_cols = ['Tenure', 'Usage Frequency', 'Support Calls', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction']


imputer = SimpleImputer(strategy='median')
imputed_values = imputer.fit_transform(raw[impute_cols])

raw[impute_cols] = imputed_values

In [9]:
raw.isna().sum()

CustomerID           0
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Customer Status      0
Last Due Date        0
Last Payment Date    0
Churn                0
dtype: int64

### Save clean dataset

In [10]:
clean_filepath = '../data/processed/train_clean.csv'
raw.to_csv(clean_filepath)
clean = raw

# Some EDA

In [11]:
clean.corr()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Customer Status,Last Due Date,Last Payment Date,Churn
CustomerID,1.000000,0.001026,-0.000320,0.001525,0.000530,0.002508,0.001633,0.000892,-0.004151,0.000676,0.001557,-0.001484,-0.001409,0.000007,0.001484
Age,0.001026,1.000000,-0.035806,-0.011601,-0.006561,0.178720,0.067048,-0.004480,-0.079027,-0.098190,0.028514,-0.189481,0.000778,0.044584,0.189481
Gender,-0.000320,-0.035806,1.000000,0.009082,0.007170,-0.085163,-0.047390,0.003664,0.058389,0.071085,0.182525,0.158337,-0.001831,-0.032480,-0.158337
Tenure,0.001525,-0.011601,0.009082,1.000000,-0.034622,-0.026362,-0.016141,-0.032295,0.015549,0.015998,-0.003840,0.019302,-0.003192,-0.013001,-0.019302
Usage Frequency,0.000530,-0.006561,0.007170,-0.034622,1.000000,-0.022104,-0.013876,0.000795,0.015138,0.019834,-0.003743,0.049018,0.000102,-0.009028,-0.049018
Support Calls,0.002508,0.178720,-0.085163,-0.026362,-0.022104,1.000000,0.161287,-0.009517,-0.184654,-0.230005,0.067643,-0.465354,-0.001889,0.104408,0.465354
Payment Delay,0.001633,0.067048,-0.047390,-0.016141,-0.013876,0.161287,1.000000,-0.004666,-0.108830,-0.135910,0.037768,-0.329147,-0.000850,0.655548,0.329147
Subscription Type,0.000892,-0.004480,0.003664,-0.032295,0.000795,-0.009517,-0.004666,1.000000,0.004423,0.006307,-0.001377,0.015577,0.001228,-0.002134,-0.015577
Contract Length,-0.004151,-0.079027,0.058389,0.015549,0.015138,-0.184654,-0.108830,0.004423,1.000000,0.156651,-0.047691,0.297749,0.000556,-0.070993,-0.297749
Total Spend,0.000676,-0.098190,0.071085,0.015998,0.019834,-0.230005,-0.135910,0.006307,0.156651,1.000000,-0.058395,0.372350,0.003725,-0.086370,-0.372350


In [ ]:
sns.catplot(data=clean, x='Total Spend', y='Contract Length', hue='Churn', palette=['green', 'red'])


KeyboardInterrupt: 